# DomainSeg

## I. Getting Started Guide

### 1. DomainSeg Model Introduction

Roadwork scenes and construction objects present a challenging edge-case scenario for self-driving cars. 
Roadwork objects can be placed dynamically and their position can be changed at a moments notice by construction workers. 
On many occassions, roadwork objects can be used to artifically narrow the driving corridor and guide vehicles to merge lanes. 
These scenarios are very demanding for self-driving cars, creating a need for a robust and reliable roadwork scene segmentation technology. 

DomainSeg addresses this key technology gap, delivering robust safety perception across urban driving scenarios, highways and even unstrcutured roads. 
It is able to adapt to challenging weather conditions such as snow and low-light, and is robust to edge cases such as detection of traffic cones that have been knocked over by other cars.

<img src="../../Media/DomainSeg_GIF.gif">

In principle, DomainSeg is a lightweight segmentation model in the VisionPilot stack that reuses a pretrained perception trunk and adds a compact decoder head for domain-specific segmentation.

Its implementation is split into three modules:

- DomainSeg Network (`Models/model_components/domain_seg_network.py`) : top-level wrapper.
- DomainSeg Upstream (`Models/model_components/domain_seg_upstream.py`) : frozen pretrained feature pipeline. 
- DomainSeg Head (`Models/model_components/domain_seg_head.py`) : trainable segmentation decoder. 

This design ensures efficient training where most heavy feature extraction is reused and frozen.
Also, decoder upsampling + skip fusion helps recover fine segmentation boundaries.



### 2. Environment Setup

**If you have done these steps of environment setup in other End-to-End Model Tutorials, please skip this and move forward to the next subsection `3. Download Models`.**

First, we need to clone the repository to your local environment. We will use the official repository from the
Autoware Foundation.

In [ ]:
import os

base_dir = os.getcwd()

# Clone Autoware VisionPilot repo, if not yet done
if not os.path.exists("./autoware_vision_pilot"):
    !git clone https://github.com/autowarefoundation/autoware_vision_pilot.git

The VisionPilot project relies on several key libraries, including **PyTorch** for model execution and
**OpenCV** for image processing.

We will use `pip` to install the required dependencies directly from the `requirements.txt` file provided in 
the `Models` directory.

In [ ]:
# Install required dependencies
%pip install --upgrade pip
%pip install -r autoware_vision_pilot/Models/requirements.txt

# Verify the installation (checking torch as a primary dependency)
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA availability: {torch.cuda.is_available()}")

### 3. Download Models

You can manually download the model weight files, using the links below. Subsequent tutorial sections assume
you already downloaded them and put them inside directory: 
`autoware_vision_pilot/Tutorials/E2E_Models/weights/`.

- [Link to Download Pytorch Model Weights (*.pth)](https://drive.google.com/file/d/1sYa2ltivJZEWMsTFZXAOaHK--Ovnadu2/view?usp=drive_link)
- [Link to Download Traced Pytorch Model (*.pt)](https://drive.google.com/file/d/12fLHpx3IZDglRJaDZT9kMhsV-f6ZTyks/view?usp=drive_link)
- [Link to Download ONNX FP32 Weights (*.onnx)](https://drive.google.com/file/d/1zCworKw4aQ9_hDBkHfj1-sXitAAebl5Y/view?usp=drive_link)

You can also use below script block, which uses `gdown` to download above model weights into such default directory.

In [ ]:
%pip install gdown

import gdown

model_dir = "./weights"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

metadata = {
    "pth"   : {
        "url"       : "https://drive.google.com/uc?id=1sYa2ltivJZEWMsTFZXAOaHK--Ovnadu2",
        "filepath"  : os.path.join(base_dir, model_dir, "domainseg.pth")
    },
    "pt"    : {
        "url"       : "https://drive.google.com/uc?id=12fLHpx3IZDglRJaDZT9kMhsV-f6ZTyks",
        "filepath"  : os.path.join(base_dir, model_dir, "domainseg.pt")
    },
    "onnx"  : {
        "url"       : "https://drive.google.com/uc?id=1zCworKw4aQ9_hDBkHfj1-sXitAAebl5Y",
        "filepath"  : os.path.join(base_dir, model_dir, "domainseg.onnx")
    }
}

for weight_type in metadata:

    url         = metadata[weight_type]["url"]
    filepath    = metadata[weight_type]["filepath"]

    if not os.path.exists(filepath):
        print(f"Downloading DomainSeg {weight_type} weights...")
        gdown.download(url, filepath, quiet = False)
    else:
        print(f"DomainSeg {weight_type} weights already exist at {filepath}. Skipping download.")

## II. Quick Test/Inference

### 1. Image Inference

Pre-requisites: please ensure that you have completed the above **Environment Setup** and **Download Models** first.

#### a. Run Batch Image Inference

The `image_visualization.py` script enables batch image processing by loading a pre-trained DomainSeg checkpoint, running inference for each image, and generating a blended segmentation overlay on the original frames.

To run this in any environment, we should navigate to the DomainSeg visualization directory so relative imports and paths resolve correctly.

Key Script Parameters:
- `-p / --model_checkpoint_path`: path to your `.pth` file containing trained DomainSeg weights.
- `-i / --input_image_dirpath`: directory with input `.png`, `.jpg`, or `.jpeg` images.
- `-o / --output_image_dirpath`: destination directory where visualization images will be saved.

In [ ]:
# 1. Path declaration
# Using the previously downloaded PyTorch DomainSeg checkpoint.
CHECKPOINT = metadata["pth"]["filepath"]
INPUT_DIR = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/images/"
)
OUTPUT_DIR = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/domainseg/images/"
)

# 2. Navigate to visualization directory and execute batch image inference
!cd autoware_vision_pilot/Models/visualizations/DomainSeg && \
python3 image_visualization.py \
    -p {os.path.abspath(CHECKPOINT)} \
    -i {os.path.abspath(INPUT_DIR)} \
    -o {os.path.abspath(OUTPUT_DIR)}

#### b. Display Results in Notebook

After running batch inference, we can visualize generated output images directly in this notebook using `matplotlib` and `PIL`.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Path to inference results generated above
result_files = sorted([
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith(".png")
])

if (result_files):
    # Display the first 3 results
    fig, axes = plt.subplots(
        1, 3,
        figsize = (20, 10)
    )
    for i, ax in enumerate(axes):
        if (i < len(result_files)):
            img_path = os.path.join(OUTPUT_DIR, result_files[i])
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(f"Result: {result_files[i]}")
            ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No result images found. Check your output directory.")

### 2. Video Inference

In this subsection, we apply DomainSeg to video streams. The `video_visualization.py` script reads frames, runs segmentation inference, and writes a blended output video.

The pipeline uses `OpenCV` for frame IO and color conversion, then overlays model predictions with transparency to produce an interpretable roadwork-domain visualization.

Key script parameters:

- `-p / --model_checkpoint_path`: path to trained DomainSeg `.pth` weights.
- `-i / --input_video_filepath`: path to input video file.
- `-o / --output_file`: output filepath prefix for generated `.avi` video.
- `-v / --vis`: optional flag to display live per-frame visualization while processing.

Technical details:

- Frames are resized to `(640, 320)` before inference to match model input expectations.
- Output is composed at HD resolution `(1280, 720)` and blended with alpha `0.5`.
- Video capture/writer handles are released automatically at completion.

In [ ]:
# 1. Path declaration
INPUT_PATH = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/videos/highway_normal.mp4"
)
OUTPUT_FILE = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/domainseg/videos/highway_normal_output"
)

# 2. Navigate to visualization directory and execute video inference
!cd autoware_vision_pilot/Models/visualizations/DomainSeg && \
python3 video_visualization.py \
    -p {os.path.abspath(CHECKPOINT)} \
    -i {os.path.abspath(INPUT_PATH)} \
    -o {os.path.abspath(OUTPUT_FILE)}

## III. Model Training

### 1. Model Training on New Data

#### a. Load pre-trained model or load vanilla un-trained model for training from scratch

#### a. Load pre-trained model or load vanilla un-trained model for training from scratch

In the current DomainSeg training code, model loading is controlled by:

- `Models/training/train_domain_seg.py`
- `Models/training/domain_seg_trainer.py`

There are currently **2 supported startup modes**:

##### i) Train DomainSeg with pretrained upstream (recommended default)  
   - Loads pretrained `SceneSegNetwork` weights into DomainSeg upstream.
   - DomainSeg head starts from random initialization.
   - Triggered when `--load_from_save` is **not** used.

##### ii) Resume / fine-tune from a saved DomainSeg checkpoint  
   - Loads full DomainSeg weights (upstream + head) from a `.pth`.
   - Triggered when `--load_from_save` is used.

---

##### Mode 1: pretrained upstream + new DomainSeg head

````bash
python3 Models/training/train_domain_seg.py \
  -r /path/to/domainseg_data_root/ \
  -s /path/to/save_dir/ \
  -m /path/to/sceneseg_pretrained.pth \
  -t /path/to/test_vis_dir/
````

##### Mode 2: resume from saved DomainSeg checkpoint

````bash
python3 Models/training/train_domain_seg.py \
  -r /path/to/domainseg_data_root/ \
  -s /path/to/save_dir/ \
  -c /path/to/domainseg_checkpoint.pth \
  -t /path/to/test_vis_dir/ \
  -l
````

##### Important note on current script behavior

In `train_domain_seg.py`, `checkpoint_path` is currently assigned from `args.pretrained_checkpoint_path` instead of 
`args.checkpoint_path`. So for clean resume behavior, this line should be corrected in code.

Also, a true "vanilla from scratch" (random upstream + random head) is not currently enabled in `DomainSegTrainer`.
It requires adding a new branch that instantiates `SceneSegNetwork()` without loading weights.

#### b. Prepare the training datasets

DomainSeg data parsing scripts are under `Models/data_parsing/DomainSeg/` and convert open datasets into a unified **binary segmentation** format for roadwork objects.

From the parser README and scripts, the unified foreground class includes movable construction objects such as:
- Traffic cones
- Traffic barrels / drums
- Tubular markers / vertical panels
- Temporary construction barriers (dataset-dependent mapping).

Supported open datasets:
- ROADWork
- Mapillary Vistas 2.0

In practice, the project mainly relies on **ROADWork** for DomainSeg training. 
Mapillary Vistas 2.0 support is retained but was found less representative for construction-zone scenarios.

##### i) ROADWork (recommended)

Script: `Models/data_parsing/DomainSeg/ROADWork/process_roadwork.py`

Expected input layout:

```text
<ROADWORK_ROOT>/
  gtFine/
  images/
```

Run from repository root:
````bash
python3 Models/data_parsing/DomainSeg/ROADWork/process_roadwork.py \\
  -d /path/to/ROADWork_raw/ \\
  -s /path/to/processed/ROADWork_Processed/
````

Output layout:
```text
<processed>/ROADWork_Processed/
  label/
  image/
  visualization/
```

Parser behavior highlights:
- Converts ROADWork label IDs `{13, 14, 15, 16}` into a binary foreground mask (`255`).
- Crops tall images to approximately 2:1 aspect ratio when needed.
- Saves audit overlays to `visualization/`.

##### ii) Mapillary Vistas 2.0 (optional)

Script: `Models/data_parsing/DomainSeg/Mapillary_Vistas_2.0/process_mapillary_vistas.py`

Expected simplified inputs (train/val images and labels separated):
- train labels folder (`*.png`), train images folder (`*.jpg`)
- validation labels folder (`*.png`), validation images folder (`*.jpg`)

Run from repository root:
````bash
python3 Models/data_parsing/DomainSeg/Mapillary_Vistas_2.0/process_mapillary_vistas.py \\
  -trlb /path/to/mapillary/train/labels \\
  -trim /path/to/mapillary/train/images \\
  -valb /path/to/mapillary/val/labels \\
  -vaim /path/to/mapillary/val/images \\
  -lbs /path/to/processed/MAPILLARY_Processed/label \\
  -ims /path/to/processed/MAPILLARY_Processed/image \\
  -vis /path/to/processed/MAPILLARY_Processed/visualization
````

Parser behavior highlights:
- Resizes image and label to `(1920, 1080)`.
- Extracts selected construction classes into binary masks.
- Filters out samples that do not contain target classes.

**Recommended workflow:**
1. Process ROADWork first and validate output masks/overlays.
2. Optionally process Mapillary Vistas for experiments.
3. Ensure output folders contain matched `image/` and `label/` files before training.

#### c. How to load dataset

`train_domain_seg.py` loads data through `LoadDataDomainSeg` from `Models/data_utils/load_data_domain_seg.py`.

Current training script expects this processed structure under `--root`:

```text
<ROOT>/
  ROADWork_Processed/
    label/
    image/
    visualization/   # optional for auditing
  Test/
```

Path construction in `train_domain_seg.py`:

````python
roadwork_labels_filepath = root + 'ROADWork_Processed/label/'
roadwork_images_filepath = root + 'ROADWork_Processed/image/'
test_images = root + 'Test/'
````

Dataset loader initialization:

````python
roadwork_Dataset = LoadDataDomainSeg(
    roadwork_labels_filepath,
    roadwork_images_filepath
)
roadwork_num_train_samples, roadwork_num_val_samples = roadwork_Dataset.getItemCount()
````

Then samples are consumed by:
- `getItemTrain(index)` during training
- `getItemVal(index)` during validation

Before starting full training, verify:

1. Every image has a corresponding mask filename/index.
2. Masks are binary (background 0, foreground 255).
3. `getItemCount()` returns non-zero train/val samples.
4. `Test/` exists if you want epoch-wise visual test outputs.

#### d. How to run training

After dataset preparation is complete and paths are configured, training is launched with `train_domain_seg.py`.

##### i) Step 1: Configure required runtime arguments

From current script behavior in `Models/training/train_domain_seg.py`, you should provide:

- `-r / --root`: root folder containing processed data (`ROADWork_Processed/...`) and `Test/`
- `-s / --model_save_root_path`: folder prefix where checkpoints are written
- `-t / --test_images_save_path`: folder prefix for epoch-wise test visualizations
- `-m / --pretrained_checkpoint_path`: upstream SceneSeg checkpoint (default training path)
- `-c / --checkpoint_path` + `-l / --load_from_save`: resume/fine-tune mode

##### ii) Step 2: Run training

From repository root:

````bash
python3 Models/training/train_domain_seg.py \\
  -r /absolute/path/to/domainseg_data_root/ \\
  -s /absolute/path/to/Models/saves/DomainSeg/models/ \\
  -m /absolute/path/to/sceneseg_pretrained.pth \\
  -t /absolute/path/to/Models/saves/DomainSeg/test_vis/
````

Resume mode:

````bash
python3 Models/training/train_domain_seg.py \\
  -r /absolute/path/to/domainseg_data_root/ \\
  -s /absolute/path/to/Models/saves/DomainSeg/models/ \\
  -c /absolute/path/to/domainseg_checkpoint.pth \\
  -t /absolute/path/to/Models/saves/DomainSeg/test_vis/ \\
  -l
````

Btw, in current script, `checkpoint_path` is assigned from `args.pretrained_checkpoint_path` instead of `args.checkpoint_path`.
Please fix this line in `train_domain_seg.py` before resume training.

##### iii) Step 3: What the training loop does

- Loads ROADWork train/val splits via `LoadDataDomainSeg`.
- Trains for `num_epochs = 50`.
- Uses BCE-with-logits loss (`nn.BCEWithLogitsLoss`) in `DomainSegTrainer.run_model()`.
- Simulates batch updates by gradient accumulation (`optimizer.step()` every `batch_size` samples).
- Applies schedules in-script:
  - batch size: 8 => 6 => 5 => 4 => 3 => 2 by epoch range
  - learning rate: `5e-4` default, then `1e-4`, then `2.5e-5`
  - augmentations enabled during mid training epochs
- Per epoch:
  - saves a `.pth` checkpoint,
  - runs `trainer.test(...)` on `Test/` images,
  - runs full validation and logs mean IoU.

#### e. How to visualize training results

DomainSeg training exposes two result channels:

1. **TensorBoard logs** from `SummaryWriter()` in `DomainSegTrainer`
2. **Saved per-epoch test visualizations** written by `DomainSegTrainer.test(...)`

##### a) Open TensorBoard

Run from repository root:

````bash
tensorboard --logdir runs --port 6006
````

Then open `http://localhost:6006`.

You should see:
- Scalar: `Loss/train` (logged every 250 train steps)
- Scalar: `Val/IoU` (logged after each validation epoch)
- Figures: `Image`, `Prediction`, `Ground Truth` (logged every 1000 train steps)

##### b) Inspect saved test visualizations

At each epoch end, the trainer runs inference on images from `<ROOT>/Test/` and saves blended overlays
to the path prefix passed by `-t`.

File naming pattern in current code: `{log_count}_{i}.jpg`

Practical checks:
1. Confirm foreground masks align with construction objects.
2. Track false positives on non-roadwork scenes.
3. Compare TensorBoard `Val/IoU` trend with qualitative test overlays.

### 2. Building an Inference Pipeline

#### a. Using ROS2 to publish an image, run inference and visualize results

TBD.

#### b. Using IceOryx to publish an image, run inference and visualize results

TBD.

#### c. Using Zenoh to publish an image, run inference and visualize results

TBD.

#### d. Using Gstreamer to get an image, run inference and visualize results

TBD.